# PyTorch, Neural Networks & Transformers — From Scratch

**Part 0 of the LLM From Scratch series.**

This notebook is the prerequisite chapter: before building a language model,
you need to be fluent in PyTorch and understand *why* each architectural
piece exists. We build up in four stages:

1. **PyTorch fundamentals** — tensors, autograd, `nn.Module`, the training loop
2. **Neural networks / DNNs** — from a single neuron to a multi-layer perceptron
3. **CNNs** — convolutions explained and implemented, then a working classifier
4. **The Transformer, piece by piece** — attention, multi-head attention,
   layer norm, feed-forward blocks, masking, and a working mini-GPT you train
   and sample from yourself

Every section follows the same pattern: **explain the idea -> implement it
manually/minimally -> show the PyTorch built-in -> use the built-in from then
on.** This is deliberate -- implementing the primitive once is what makes the
built-in stop feeling like a black box.

**Prerequisites:** basic Python, basic linear algebra (matrix multiplication,
dot products), and basic calculus (what a derivative is). No prior PyTorch
or deep learning experience assumed.


## Setup

Run this once. If you don't have a GPU, everything in this notebook still
runs fine on CPU -- the models here are intentionally tiny.


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torch.amp import autocast, GradScaler  # Mixed Precision Training
from torchvision import transforms
from torchvision.datasets import ImageFolder  # or MNIST, CIFAR10, etc.
from dataclasses import dataclass
import math
import time

import matplotlib.pyplot as plt

torch.manual_seed(42)

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

Using device: cpu
PyTorch version: 2.13.0+cpu


---
# Part 1 -- PyTorch Fundamentals

PyTorch's whole job is two things:

1. **Tensors** -- N-dimensional arrays (like NumPy) that can live on a GPU
2. **Autograd** -- automatic differentiation, so you never hand-derive a
   gradient for a real model

Everything else in the library (`nn.Module`, optimizers, `DataLoader`) is
convenience built on top of those two ideas.


## 1.1 Tensors

A tensor is just an array with a shape and a dtype. Scalars are 0-D tensors,
vectors are 1-D, matrices are 2-D, and a batch of RGB images is a 4-D tensor
`(batch, channels, height, width)`.


In [3]:
# Creating tensors
scalar = torch.tensor(5.0)
vector = torch.tensor([1.0, 2.0, 3.0])
matrix = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
batch_of_images = torch.zeros(8, 3, 32, 32)  # 8 RGB images, 32x32

for name, t in [("scalar", scalar), ("vector", vector), ("matrix", matrix), ("batch_of_images", batch_of_images)]:
    print(f"{name:16s} shape={tuple(t.shape)} ndim={t.ndim} dtype={t.dtype}")

scalar           shape=() ndim=0 dtype=torch.float32
vector           shape=(3,) ndim=1 dtype=torch.float32
matrix           shape=(2, 2) ndim=2 dtype=torch.float32
batch_of_images  shape=(8, 3, 32, 32) ndim=4 dtype=torch.float32


In [4]:
# Common constructors
print(torch.zeros(2, 3))
print(torch.ones(2, 3))
print(torch.randn(2, 3))         # standard normal
print(torch.arange(0, 10, 2))    # like range()
print(torch.linspace(0, 1, 5))   # 5 evenly spaced points

# From/to NumPy (shares memory when on CPU!)
import numpy as np
np_array = np.array([1, 2, 3])
t_from_np = torch.from_numpy(np_array)
print(t_from_np, t_from_np.dtype)


tensor([[0., 0., 0.],
        [0., 0., 0.]])
tensor([[1., 1., 1.],
        [1., 1., 1.]])
tensor([[ 0.3367,  0.1288,  0.2345],
        [ 0.2303, -1.1229, -0.1863]])
tensor([0, 2, 4, 6, 8])
tensor([0.0000, 0.2500, 0.5000, 0.7500, 1.0000])
tensor([1, 2, 3], dtype=torch.int32) torch.int32


### Shape manipulation

This is the single most common source of bugs in deep learning code. Get
comfortable with these four operations now -- you will use them constantly.


In [5]:
x = torch.arange(12)
print("original:", x.shape)

# reshape / view -- view requires contiguous memory, reshape doesn't care
x2 = x.view(3, 4)
print("view(3,4):", x2.shape)

# unsqueeze -- add a dimension of size 1 (e.g. turn a vector into a batch of 1)
x3 = x.unsqueeze(0)
print("unsqueeze(0):", x3.shape)

# squeeze -- remove dimensions of size 1
x4 = x3.squeeze(0)
print("squeeze(0):", x4.shape)

# permute / transpose -- reorder dimensions (crucial for attention later)
m = torch.randn(2, 3, 4)  # e.g. (batch, seq_len, features)
m_t = m.transpose(1, 2)   # swap seq_len and features -> (batch, features, seq_len)
print("transpose(1,2):", m_t.shape)

original: torch.Size([12])
view(3,4): torch.Size([3, 4])
unsqueeze(0): torch.Size([1, 12])
squeeze(0): torch.Size([12])
transpose(1,2): torch.Size([2, 4, 3])


In [6]:
m

tensor([[[-0.2483, -1.2082, -0.4777,  0.5201],
         [ 1.6423, -0.1596, -0.4974,  0.4396],
         [ 0.3189, -0.4245,  0.3057, -0.7746]],

        [[ 0.0349,  0.3211,  1.5736, -0.8455],
         [-1.2742,  2.1228, -1.2347, -0.4879],
         [-1.4181,  0.8963,  0.0499,  2.2667]]])

In [7]:
m_t

tensor([[[-0.2483,  1.6423,  0.3189],
         [-1.2082, -0.1596, -0.4245],
         [-0.4777, -0.4974,  0.3057],
         [ 0.5201,  0.4396, -0.7746]],

        [[ 0.0349, -1.2742, -1.4181],
         [ 0.3211,  2.1228,  0.8963],
         [ 1.5736, -1.2347,  0.0499],
         [-0.8455, -0.4879,  2.2667]]])

In [8]:
# Broadcasting: PyTorch automatically expands smaller tensors to match shapes
a = torch.tensor([[1.0, 2.0, 3.0],
                   [4.0, 5.0, 6.0]])       # shape (2, 3)
b = torch.tensor([10.0, 20.0, 30.0])       # shape (3,)

print(a + b)  # b is broadcast across both rows
print()
print("a shape:", a.shape, "b shape:", b.shape, "-> result shape:", (a + b).shape)

tensor([[11., 22., 33.],
        [14., 25., 36.]])

a shape: torch.Size([2, 3]) b shape: torch.Size([3]) -> result shape: torch.Size([2, 3])


### Matrix multiplication

This is the operation everything in deep learning reduces to. Know the
difference between element-wise `*` and matrix multiplication `@`.

In [9]:
A = torch.randn(2, 3)
B = torch.randn(3, 4)

C = A @ B                 # matrix multiply, same as torch.matmul(A, B)
print("A @ B shape:", C.shape)   # (2, 4)

# Element-wise multiply requires matching (or broadcastable) shapes
D = torch.randn(2, 3)
E = A * D                 # element-wise, NOT matrix multiply
print("A * D shape:", E.shape)   # (2, 3)

# batched matmul -- this is what attention uses constantly
batch_A = torch.randn(8, 2, 3)   # 8 matrices of shape (2,3)
batch_B = torch.randn(8, 3, 4)   # 8 matrices of shape (3,4)
batch_C = batch_A @ batch_B
print("batched matmul shape:", batch_C.shape)  # (8, 2, 4)

A @ B shape: torch.Size([2, 4])
A * D shape: torch.Size([2, 3])
batched matmul shape: torch.Size([8, 2, 4])


## 1.2 Autograd -- automatic differentiation

Set `requires_grad=True` on a tensor and PyTorch builds a computation graph
as you use it. Call `.backward()` on a scalar output and PyTorch walks the
graph backward, computing d(output)/d(input) for every tensor that required
a gradient -- this is the chain rule, applied automatically.